# 12b. hard_v2 — cleaned name-only entity completion, three systems (fixed v4: verified dataset signature)

This version fixes the Colab/Hugging Face stall and makes the long decode genuinely resume-safe.

**Before running:** if the old notebook is still hanging, use **Runtime → Disconnect and delete runtime**, open this notebook, select an A100/GPU runtime, and run from the first cell.

### What is fixed
1. `facebook/bart-base` is downloaded **once** with resumable `curl`, verified by file size, and then loaded only from local files. It no longer starts two separate Hub downloads.
2. `HF_TOKEN` is optional because BART-base is public. `hf-xet` is disabled/removed to avoid the observed zero-byte transfer stall.
3. All checkpoint and data paths are auto-discovered recursively in Drive and validated before the expensive run begins.
4. Every prediction pass is saved atomically every 25 examples and resumes from the last completed example after a disconnect.
5. Results are written to a fresh versioned directory, `hard_v2_eval_fixed_v4/`, so partial files from the earlier notebook cannot be mixed with this run.
6. The final cell creates one ZIP file for easy upload.

**Experiment:** BART baseline_v2, fusion_only, and full_dual are each decoded with `off` and cleaned `hard_v2`. The held-out development check also evaluates positive-only α=1 and score-gated hard_v2.

7. If the fair baseline model is missing, the notebook searches Trainer `checkpoint-*` directories and then automatically retrains the exact Notebook 1b baseline configuration instead of stopping.

8. Dataset discovery now opens every candidate `webnlg_processed.pkl` and accepts only the original split signature (train=13,211, dev=1,667, test=5,713). It prints all rejected candidate paths and sizes instead of silently selecting the wrong preprocessing.


In [ ]:
# 1. Fresh-runtime bootstrap. Run this cell FIRST, before any transformers/HF import.
import os

# These variables must exist before huggingface_hub/transformers are imported.
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '600'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Keep the stack compatible with the existing project code and remove the transfer
# backend that produced the 0-byte model.safetensors stall.
!pip -q install "transformers>=4.44,<5" "huggingface_hub>=0.25,<1" datasets nltk rouge-score accelerate sentencepiece sacremoses sacrebleu safetensors requests tqdm
!pip -q uninstall -y hf-xet >/dev/null 2>&1 || true

import sys, json, pickle, random, time, shutil, subprocess, tempfile
from pathlib import Path
import numpy as np
import torch
from google.colab import drive, userdata

drive.mount('/content/drive')

# Optional: public BART-base does not require authentication. If a valid Colab
# secret is present, other private Hub calls can still use it.
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF token found (kept private).')
else:
    print('No HF token found; this is fine for public facebook/bart-base.')

print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'A GPU runtime is required. In Colab: Runtime → Change runtime type → GPU.'


In [ ]:
# 1.1 PyTorch Geometric.
try:
    import torch_geometric
    print('torch_geometric ok:', torch_geometric.__version__)
except Exception:
    tv = torch.__version__.split('+')[0]
    cv = torch.version.cuda
    wheel_tag = 'cpu' if cv is None else f'cu{cv.replace(".", "")}'
    url = f'https://data.pyg.org/whl/torch-{tv}+{wheel_tag}.html'
    print('Installing PyG wheels from:', url)
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch-geometric', 'torch-scatter', 'torch-sparse', '-f', url
    ])
    import torch_geometric
    print('torch_geometric installed:', torch_geometric.__version__)


In [ ]:
# 2. Shared module + robust Drive path discovery.
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
PREFERRED_PROJECT_DIR = DRIVE_ROOT / 'kg_llm_project'
PROJECT_DIR = str(PREFERRED_PROJECT_DIR)

MODEL_WEIGHT_NAMES = (
    'model.safetensors',
    'pytorch_model.bin',
)

def _existing_file(candidates):
    for path in candidates:
        path = Path(path)
        if path.is_file():
            return str(path)
    return None

def _is_bart_model_dir(path):
    path = Path(path)
    if not path.is_dir():
        return False

    config_path = path / 'config.json'
    if not config_path.is_file():
        return False

    if not any(
        (path / weight_name).is_file()
        for weight_name in MODEL_WEIGHT_NAMES
    ):
        return False

    try:
        config = json.load(
            open(config_path, encoding='utf-8')
        )
    except Exception:
        return False

    return config.get('model_type') == 'bart'

def _existing_model_dir(candidates):
    for path in candidates:
        if _is_bart_model_dir(path):
            return str(Path(path))
    return None

def _rank_path(path, hints):
    path = Path(path)
    text = str(path).lower()
    score = sum(
        10 for hint in hints
        if hint.lower() in text
    )

    if path.name.startswith('checkpoint-'):
        try:
            score += (
                int(path.name.split('-')[-1])
                / 100000
            )
        except Exception:
            pass

    return score - len(path.parts) * 0.001

def _find_file(filename, roots, hints=()):
    found = []

    for root in roots:
        root = Path(root)
        if not root.exists():
            continue

        try:
            found.extend(root.rglob(filename))
        except Exception as exc:
            print(
                f'Warning: could not fully search '
                f'{root}: {exc}'
            )

    found = [
        path for path in found
        if path.is_file()
    ]

    if not found:
        return None

    return str(
        max(
            found,
            key=lambda path: _rank_path(
                path,
                hints,
            ),
        )
    )

def _processed_signature(directory):
    """Return split sizes for a candidate processed directory."""
    directory = Path(directory)
    data_path = directory / 'webnlg_processed.pkl'
    vocab_path = directory / 'vocabularies.pkl'

    if not data_path.is_file() or not vocab_path.is_file():
        return None

    try:
        with open(data_path, 'rb') as handle:
            candidate_data = pickle.load(handle)
    except Exception as exc:
        return {
            'path': str(directory),
            'valid_pickle': False,
            'error': repr(exc),
        }

    if not isinstance(candidate_data, dict):
        return {
            'path': str(directory),
            'valid_pickle': False,
            'error': (
                'webnlg_processed.pkl is not a split dictionary'
            ),
        }

    split_sizes = {
        split: len(candidate_data.get(split, []))
        for split in ('train', 'dev', 'test')
    }

    return {
        'path': str(directory),
        'valid_pickle': True,
        'sizes': split_sizes,
        'has_vocab': vocab_path.is_file(),
        'graph_count': sum(
            (directory / f'graphs_{split}.pkl').is_file()
            for split in ('train', 'dev', 'test')
        ),
    }

def _find_processed_dir(roots):
    """
    Select only the original processed WebNLG dataset used by notebooks 9/12
    and the fair baseline. Do not silently accept another WebNLG preprocessing.
    """
    expected_sizes = {
        'train': 13211,
        'dev': 1667,
        'test': 5713,
    }

    candidate_dirs = []
    seen = set()

    preferred = [
        PREFERRED_PROJECT_DIR
        / 'baseline-bart-webnlg'
        / 'processed',
        PREFERRED_PROJECT_DIR
        / 'baseline_bart_webnlg'
        / 'processed',
        PREFERRED_PROJECT_DIR
        / 'processed',
    ]

    for directory in preferred:
        key = str(directory)
        if key not in seen:
            seen.add(key)
            candidate_dirs.append(directory)

    for root in roots:
        root = Path(root)
        if not root.exists():
            continue

        try:
            for data_path in root.rglob('webnlg_processed.pkl'):
                directory = data_path.parent
                key = str(directory)
                if key not in seen:
                    seen.add(key)
                    candidate_dirs.append(directory)
        except Exception as exc:
            print(
                f'Warning: could not fully search '
                f'{root}: {exc}'
            )

    reports = []
    for directory in candidate_dirs:
        report = _processed_signature(directory)
        if report is not None:
            reports.append(report)

    print('Processed-dataset candidates found:')
    if not reports:
        print('  none')
    else:
        for report in reports:
            if report.get('valid_pickle'):
                print(
                    '  ',
                    report['path'],
                    '| sizes =',
                    report['sizes'],
                    '| graph caches =',
                    report['graph_count'],
                    '/3',
                )
            else:
                print(
                    '  ',
                    report['path'],
                    '| unreadable:',
                    report.get('error'),
                )

    exact = [
        report
        for report in reports
        if (
            report.get('valid_pickle')
            and report.get('sizes') == expected_sizes
        )
    ]

    if not exact:
        formatted = '\n'.join(
            (
                f"- {report['path']}: "
                f"{report.get('sizes', report.get('error'))}"
            )
            for report in reports
        ) or '- no webnlg_processed.pkl candidates found'

        raise FileNotFoundError(
            'The original processed WebNLG dataset was not found. '
            'Required split signature: '
            f'{expected_sizes}. Candidates inspected:\n'
            f'{formatted}\n\n'
            'Do not remove this check or use a different processed file: '
            'the checkpoints and all prior 2,510-input evaluations were '
            'built from the original dataset. Restore the original files to '
            '/content/drive/MyDrive/kg_llm_project/'
            'baseline-bart-webnlg/processed/.'
        )

    def exact_score(report):
        directory = Path(report['path'])
        return (
            report['graph_count'] * 100
            + _rank_path(
                directory,
                (
                    'kg_llm_project',
                    'baseline-bart-webnlg',
                    'processed',
                ),
            )
        )

    selected = max(exact, key=exact_score)

    print(
        'Selected original processed dataset:',
        selected['path'],
        '| signature =',
        selected['sizes'],
    )

    return selected['path']

def _find_fair_baseline_model(roots):
    preferred = [
        PREFERRED_PROJECT_DIR
        / 'baseline-bart-webnlg'
        / 'checkpoints'
        / 'baseline_bart_v2_fair',
        PREFERRED_PROJECT_DIR
        / 'baseline_bart_webnlg'
        / 'checkpoints'
        / 'baseline_bart_v2_fair',
    ]

    direct = _existing_model_dir(preferred)
    if direct is not None:
        return direct, [direct]

    candidates = []
    seen = set()

    for root in roots:
        root = Path(root)
        if not root.exists():
            continue

        try:
            config_paths = list(
                root.rglob('config.json')
            )
        except Exception as exc:
            print(
                f'Warning: could not fully search '
                f'{root}: {exc}'
            )
            config_paths = []

        for config_path in config_paths:
            directory = config_path.parent
            text = str(directory).lower()

            if not any(
                hint in text
                for hint in (
                    'baseline',
                    'baseline_bart',
                    'bart_v2',
                    'fair',
                )
            ):
                continue

            key = str(directory)
            if key in seen:
                continue
            seen.add(key)

            if _is_bart_model_dir(directory):
                candidates.append(directory)

    candidates = sorted(
        candidates,
        key=lambda path: _rank_path(
            path,
            (
                'baseline_bart_v2_fair',
                'baseline-bart-webnlg',
                'baseline',
                'fair',
            ),
        ),
        reverse=True,
    )

    if not candidates:
        return None, []

    return (
        str(candidates[0]),
        [
            str(path)
            for path in candidates[:10]
        ],
    )

SEARCH_ROOTS = [
    PREFERRED_PROJECT_DIR
]
if not PREFERRED_PROJECT_DIR.exists():
    SEARCH_ROOTS = [
        DRIVE_ROOT
    ]

COMMON_SRC = _existing_file([
    PREFERRED_PROJECT_DIR
    / 'fixed_ablation_common.py',
]) or _find_file(
    'fixed_ablation_common.py',
    SEARCH_ROOTS,
    hints=('kg_llm_project',),
)

if (
    COMMON_SRC is None
    and PREFERRED_PROJECT_DIR.exists()
):
    COMMON_SRC = _find_file(
        'fixed_ablation_common.py',
        [DRIVE_ROOT],
        hints=('kg_llm_project',),
    )

if COMMON_SRC is None:
    raise FileNotFoundError(
        'Could not find fixed_ablation_common.py '
        'anywhere in MyDrive.'
    )

common_parent = Path(
    COMMON_SRC
).parent

if (
    not PREFERRED_PROJECT_DIR.exists()
    and common_parent.exists()
):
    PROJECT_DIR = str(common_parent)
else:
    PROJECT_DIR = str(
        PREFERRED_PROJECT_DIR
    )

shutil.copy2(
    COMMON_SRC,
    '/content/fixed_ablation_common.py',
)

if '/content' not in sys.path:
    sys.path.insert(0, '/content')

if 'fixed_ablation_common' in sys.modules:
    del sys.modules[
        'fixed_ablation_common'
    ]

import fixed_ablation_common as fac
from fixed_ablation_common import (
    load_pretrained_bart,
    load_artifacts,
    FusionOnlyGNNModel,
    FullDualOutputGNNModel,
    load_variant_checkpoint_resume,
    signed_entity_scores,
    add_final_logits_bias,
    pad_kg_nodes,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

project_path = Path(
    PROJECT_DIR
)

search_roots = [
    project_path
]

if project_path != DRIVE_ROOT:
    search_roots.append(
        DRIVE_ROOT
    )

PROCESSED_DIR = _find_processed_dir(
    search_roots
)

if PROCESSED_DIR is None:
    raise FileNotFoundError(
        'Could not find a directory containing '
        'both webnlg_processed.pkl and '
        'vocabularies.pkl anywhere in MyDrive.'
    )

CKPT_FULL = _existing_file([
    project_path
    / 'full_dual_output_outputs'
    / 'checkpoints'
    / 'full_dual_output'
    / 'model_best.pt',
]) or _find_file(
    'model_best.pt',
    search_roots,
    hints=(
        'full_dual_output',
        'full_dual',
    ),
)

CKPT_FUSION = _existing_file([
    project_path
    / 'fusion_only_outputs'
    / 'checkpoints'
    / 'fusion_only'
    / 'model_best.pt',
]) or _find_file(
    'model_best.pt',
    search_roots,
    hints=(
        'fusion_only',
        'fusion',
    ),
)

(
    BASELINE_FOUND,
    BASELINE_CANDIDATES,
) = _find_fair_baseline_model(
    search_roots
)

BASELINE_REBUILD_DIR = (
    project_path
    / 'baseline-bart-webnlg'
    / 'checkpoints'
    / 'baseline_bart_v2_fair_rebuilt'
)

if BASELINE_FOUND is None:
    BASELINE_V2 = str(
        BASELINE_REBUILD_DIR
    )
    BASELINE_NEEDS_TRAINING = True
else:
    BASELINE_V2 = BASELINE_FOUND
    BASELINE_NEEDS_TRAINING = False

missing_required = {
    'full-dual checkpoint': CKPT_FULL,
    'fusion checkpoint': CKPT_FUSION,
}

missing_labels = [
    label
    for label, value
    in missing_required.items()
    if value is None
]

if missing_labels:
    raise FileNotFoundError(
        'Drive path auto-discovery could not find: '
        + ', '.join(missing_labels)
    )

NB12_EVAL = str(
    project_path
    / 'constraint_rescue_eval'
)

BASE_EVAL = str(
    project_path
    / 'baseline_v2_eval'
)

RUN_SIGNATURE = (
    'hard_v2_name_only_local_bart_'
    'v5_baseline_fallback'
)

EVAL_OUT = str(
    project_path
    / 'hard_v2_eval_fixed_v4'
)

os.makedirs(
    EVAL_OUT,
    exist_ok=True,
)

DEVICE = 'cuda'
MAX_GEN_LEN = 128
MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 128
LOCK_MIN_DEPTH = 2
ESCAPE_MARGIN = 10.0
RUN_HELDOUT_VALIDATION = True
HELDOUT_N = 500
SAVE_EVERY = 25

BASELINE_EPOCHS = 10
BASELINE_BATCH_SIZE = 32
BASELINE_LR = 3e-5

processed_path = Path(
    PROCESSED_DIR
)

GRAPH_FILES_MISSING = [
    str(
        processed_path
        / f'graphs_{split}.pkl'
    )
    for split in (
        'train',
        'dev',
        'test',
    )
    if not (
        processed_path
        / f'graphs_{split}.pkl'
    ).is_file()
]

def atomic_json_dump(
    obj,
    path,
    **kwargs,
):
    path = str(path)
    temporary_path = path + '.tmp'

    with open(
        temporary_path,
        'w',
        encoding='utf-8',
    ) as handle:
        json.dump(
            obj,
            handle,
            ensure_ascii=False,
            **kwargs,
        )

    os.replace(
        temporary_path,
        path,
    )

run_meta_path = os.path.join(
    EVAL_OUT,
    'run_config.json',
)

run_meta = {
    'signature': RUN_SIGNATURE,
    'seed': SEED,
    'processed_dir': PROCESSED_DIR,
    'expected_split_sizes': {
        'train': 13211,
        'dev': 1667,
        'test': 5713,
    },
    'full_checkpoint': CKPT_FULL,
    'fusion_checkpoint': CKPT_FUSION,
    'baseline_v2': BASELINE_V2,
    'baseline_needs_training': (
        BASELINE_NEEDS_TRAINING
    ),
    'baseline_epochs': (
        BASELINE_EPOCHS
    ),
    'baseline_batch_size': (
        BASELINE_BATCH_SIZE
    ),
    'baseline_lr': BASELINE_LR,
    'lock_min_depth': (
        LOCK_MIN_DEPTH
    ),
    'escape_margin': (
        ESCAPE_MARGIN
    ),
}

if os.path.exists(
    run_meta_path
):
    old_meta = json.load(
        open(
            run_meta_path,
            encoding='utf-8',
        )
    )

    if (
        old_meta.get('signature')
        != RUN_SIGNATURE
    ):
        raise RuntimeError(
            f'{EVAL_OUT} contains outputs '
            'from another configuration.'
        )
else:
    atomic_json_dump(
        run_meta,
        run_meta_path,
        indent=2,
    )

print('Resolved project files:')
print(
    '  PROJECT_DIR             =',
    PROJECT_DIR,
)
print(
    '  COMMON_SRC              =',
    COMMON_SRC,
)
print(
    '  PROCESSED_DIR           =',
    PROCESSED_DIR,
)
print(
    '  CKPT_FULL               =',
    CKPT_FULL,
)
print(
    '  CKPT_FUSION             =',
    CKPT_FUSION,
)
print(
    '  BASELINE_V2             =',
    BASELINE_V2,
)
print(
    '  BASELINE_NEEDS_TRAINING =',
    BASELINE_NEEDS_TRAINING,
)
print(
    '  EVAL_OUT                =',
    EVAL_OUT,
)

if BASELINE_CANDIDATES:
    print(
        'Valid fair-baseline candidates found:'
    )
    for candidate in BASELINE_CANDIDATES:
        print(
            '   ',
            candidate,
        )
else:
    print(
        'No saved fair baseline weights were '
        'found. Cell 4 will retrain the exact '
        'fair baseline automatically.'
    )

if GRAPH_FILES_MISSING:
    print(
        'Missing graph caches will be rebuilt '
        'in Cell 4:'
    )
    for path in GRAPH_FILES_MISSING:
        print(
            '   ',
            path,
        )
else:
    print(
        'All cached graph files found.'
    )


In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive/kg_llm_project"):
    if "webnlg_processed.pkl" in files:
        print(root)

In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive/kg_llm_project"):
    if "model_best.pt" in files:
        print(root)

In [ ]:
import os

print(os.path.exists("/content/webnlg_processed.pkl"))

print(os.path.exists("/content/baseline-bart-webnlg"))

print(os.path.exists("/content/processed"))

In [ ]:
!find /content -name "webnlg_processed.pkl"

In [ ]:
import os

keywords = [
    "preprocess",
    "prepare",
    "dataset",
    "graph",
    "build",
    "process",
    "webnlg",
]

for root, dirs, files in os.walk("/content/drive/MyDrive/kg_llm_project"):
    for f in files:
        lower = f.lower()
        if any(k in lower for k in keywords):
            print(os.path.join(root, f))

In [ ]:
import pickle

with open(
    "/content/drive/MyDrive/kg_llm_project/baseline-bart-webnlg_debug/processed/webnlg_processed.pkl",
    "rb",
) as f:
    data = pickle.load(f)

print(type(data))

if isinstance(data, dict):
    print(data.keys())

    for k, v in data.items():
        try:
            print(k, len(v))
        except:
            print(k, type(v))

In [ ]:
!find /content/drive/MyDrive -name graphs_train.pkl

In [ ]:
!find /content/drive/MyDrive -name vocabularies.pkl

In [ ]:
!find /content/drive/MyDrive -name webnlg_processed.pkl

In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "webnlg_processed.pkl" in files:
        print(root)

In [ ]:
# 3. Download facebook/bart-base ONCE with resumable curl, then verify it.
# This bypasses the Hugging Face/Xet code path that stalled at 0.00/558 MB.
from urllib.parse import quote

BART_REPO = 'facebook/bart-base'
BART_LOCAL = '/content/bart-base-local'
os.makedirs(BART_LOCAL, exist_ok=True)

# Minimum sizes catch Git-LFS pointer files and zero-byte/incomplete downloads.
BART_FILES = {
    'config.json': 1_000,
    'vocab.json': 800_000,
    'merges.txt': 400_000,
    'tokenizer.json': 1_000_000,
    'model.safetensors': 500_000_000,
}


def _valid_file(path, min_bytes):
    return os.path.isfile(path) and os.path.getsize(path) >= min_bytes


def _curl_download(filename, min_bytes):
    dest = os.path.join(BART_LOCAL, filename)
    if _valid_file(dest, min_bytes):
        print(f'cached: {filename} ({os.path.getsize(dest)/1e6:.1f} MB)')
        return
    if os.path.exists(dest):
        os.remove(dest)

    part = dest + '.part'
    url = f'https://huggingface.co/{BART_REPO}/resolve/main/{quote(filename)}?download=true'
    base_cmd = [
        'curl', '-L', '--fail', '--show-error',
        '--retry', '12', '--retry-delay', '5', '--retry-all-errors',
        '--connect-timeout', '30', '--speed-time', '90', '--speed-limit', '1024',
    ]
    cmd = base_cmd + ['-C', '-', '-o', part, url]
    print(f'downloading: {filename}')
    result = subprocess.run(cmd)
    if result.returncode != 0:
        # Some mirrors do not accept Range requests. Retry once from zero.
        if os.path.exists(part):
            os.remove(part)
        result = subprocess.run(base_cmd + ['-o', part, url])
    if result.returncode != 0:
        raise RuntimeError(f'curl failed for {filename}. Re-run this cell; partial files are resumable.')
    if not _valid_file(part, min_bytes):
        size = os.path.getsize(part) if os.path.exists(part) else 0
        raise RuntimeError(f'{filename} is incomplete ({size} bytes). Re-run this cell.')
    os.replace(part, dest)
    print(f'complete: {filename} ({os.path.getsize(dest)/1e6:.1f} MB)')


for filename, min_bytes in BART_FILES.items():
    _curl_download(filename, min_bytes)

missing = [f for f, n in BART_FILES.items() if not _valid_file(os.path.join(BART_LOCAL, f), n)]
if missing:
    raise RuntimeError(f'BART local snapshot failed verification: {missing}')

print('Verified local BART snapshot:', BART_LOCAL)


In [ ]:
# 4. Load data/models. Rebuild the fair baseline if its saved weights are absent.
from transformers import (
    BartForConditionalGeneration,
    BartTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from transformers.trainer_utils import (
    get_last_checkpoint,
)
from datasets import Dataset
from torch_geometric.data import Data

def _normalise_triple(triple):
    if isinstance(triple, dict):
        return (
            str(triple['subject']),
            str(triple['predicate']),
            str(triple['object']),
        )

    return (
        str(triple[0]),
        str(triple[1]),
        str(triple[2]),
    )

@torch.no_grad()
def _initialise_node_features(
    entity_names,
    tokenizer_for_graphs,
    bart_model,
):
    embedding = (
        bart_model
        .model
        .shared
    )
    device = (
        embedding
        .weight
        .device
    )
    features = []

    for name in entity_names:
        clean_name = (
            str(name)
            .replace('_', ' ')
        )

        tokenized = tokenizer_for_graphs(
            clean_name,
            return_tensors='pt',
            add_special_tokens=False,
            truncation=True,
            max_length=32,
        )

        token_ids = (
            tokenized['input_ids']
            .to(device)
        )

        if token_ids.numel() == 0:
            pooled = torch.zeros(
                embedding.embedding_dim,
                device=device,
            )
        else:
            pooled = (
                embedding(token_ids)
                .squeeze(0)
                .mean(dim=0)
            )

        features.append(
            pooled
            .detach()
            .cpu()
        )

    if not features:
        return torch.zeros(
            (
                0,
                embedding.embedding_dim,
            ),
            dtype=torch.float,
        )

    return torch.stack(
        features,
        dim=0,
    )

def _triples_to_graph(
    example,
    relation_vocab,
    tokenizer_for_graphs,
    bart_model,
):
    triples = [
        _normalise_triple(triple)
        for triple in example['triples']
    ]

    entities = []

    for subject, _, obj in triples:
        if subject not in entities:
            entities.append(subject)
        if obj not in entities:
            entities.append(obj)

    local_index = {
        entity: index
        for index, entity
        in enumerate(entities)
    }

    source_nodes = []
    target_nodes = []
    edge_types = []
    number_of_relations = len(
        relation_vocab
    )

    for (
        subject,
        predicate,
        obj,
    ) in triples:
        if predicate not in relation_vocab:
            continue

        subject_index = local_index[
            subject
        ]
        object_index = local_index[
            obj
        ]
        relation_index = relation_vocab[
            predicate
        ]

        source_nodes.extend(
            (
                subject_index,
                object_index,
            )
        )
        target_nodes.extend(
            (
                object_index,
                subject_index,
            )
        )
        edge_types.extend(
            (
                relation_index,
                relation_index
                + number_of_relations,
            )
        )

    if source_nodes:
        edge_index = torch.tensor(
            [
                source_nodes,
                target_nodes,
            ],
            dtype=torch.long,
        )
    else:
        edge_index = torch.empty(
            (2, 0),
            dtype=torch.long,
        )

    if edge_types:
        edge_type = torch.tensor(
            edge_types,
            dtype=torch.long,
        )
    else:
        edge_type = torch.empty(
            (0,),
            dtype=torch.long,
        )

    graph = Data(
        x=_initialise_node_features(
            entities,
            tokenizer_for_graphs,
            bart_model,
        ),
        edge_index=edge_index,
        edge_type=edge_type,
        num_nodes=len(entities),
    )

    graph.entity_names = entities
    return graph

def _ensure_graph_pickles(
    processed_dir,
):
    processed_dir = Path(
        processed_dir
    )

    missing = [
        split
        for split in (
            'train',
            'dev',
            'test',
        )
        if not (
            processed_dir
            / f'graphs_{split}.pkl'
        ).is_file()
    ]

    if not missing:
        print(
            'Graph caches already complete.'
        )
        return

    print(
        'Rebuilding missing graph caches:',
        missing,
    )

    (
        tokenizer_for_graphs,
        temporary_bart,
    ) = load_pretrained_bart(
        BART_LOCAL,
        device=DEVICE,
        train_bart=False,
    )

    with open(
        processed_dir
        / 'webnlg_processed.pkl',
        'rb',
    ) as handle:
        raw_data = pickle.load(
            handle
        )

    with open(
        processed_dir
        / 'vocabularies.pkl',
        'rb',
    ) as handle:
        raw_vocab = pickle.load(
            handle
        )

    actual_sizes = {
        split: len(raw_data.get(split, []))
        for split in ('train', 'dev', 'test')
    }
    expected_sizes = {
        'train': 13211,
        'dev': 1667,
        'test': 5713,
    }

    if actual_sizes != expected_sizes:
        raise ValueError(
            'Dataset signature changed after path selection. '
            f'Expected {expected_sizes}, got {actual_sizes} from '
            f'{processed_dir}. Stop here rather than rebuilding '
            'graphs for an incompatible dataset.'
        )

    relation_vocab = raw_vocab[
        'relation_vocab'
    ]

    for split in missing:
        output_path = (
            processed_dir
            / f'graphs_{split}.pkl'
        )
        examples = raw_data[
            split
        ]
        graphs_for_split = []

        for index, example in enumerate(
            examples
        ):
            graphs_for_split.append(
                _triples_to_graph(
                    example,
                    relation_vocab,
                    tokenizer_for_graphs,
                    temporary_bart,
                )
            )

            if (
                (index + 1) % 1000 == 0
                or (index + 1)
                == len(examples)
            ):
                print(
                    f'  {split}: '
                    f'{index + 1}/'
                    f'{len(examples)}'
                )

        temporary_path = (
            str(output_path)
            + '.tmp'
        )

        with open(
            temporary_path,
            'wb',
        ) as handle:
            pickle.dump(
                graphs_for_split,
                handle,
                protocol=(
                    pickle
                    .HIGHEST_PROTOCOL
                ),
            )

        os.replace(
            temporary_path,
            output_path,
        )

        print(
            'Saved:',
            output_path,
        )

    del temporary_bart
    torch.cuda.empty_cache()

_ensure_graph_pickles(
    PROCESSED_DIR
)

data, graphs, vocab = load_artifacts(
    PROCESSED_DIR
)

num_relations = (
    len(vocab['relation_vocab'])
    * 2
)

print(
    'Loaded data:',
    {
        split: len(examples)
        for split, examples
        in data.items()
    },
)

print(
    'Loaded graphs:',
    {
        split: len(split_graphs)
        for split, split_graphs
        in graphs.items()
    },
)

assert all(
    len(data[split])
    == len(graphs[split])
    for split in (
        'train',
        'dev',
        'test',
    )
)

def _train_or_load_fair_baseline():
    baseline_path = Path(
        BASELINE_V2
    )

    baseline_path.mkdir(
        parents=True,
        exist_ok=True,
    )

    if _is_bart_model_dir(
        baseline_path
    ):
        print(
            'Loading saved fair baseline:',
            baseline_path,
        )

        return (
            BartForConditionalGeneration
            .from_pretrained(
                str(baseline_path),
                local_files_only=True,
                low_cpu_mem_usage=True,
            )
            .to(DEVICE)
        )

    print(
        'No loadable fair-baseline model '
        'was found.'
    )
    print(
        'Rebuilding it with the exact '
        'Notebook 1b configuration: '
        '10 epochs, batch size 32, '
        'constant LR 3e-5, plain CE, '
        'best by dev loss.'
    )

    baseline_tokenizer = (
        BartTokenizer
        .from_pretrained(
            BART_LOCAL,
            local_files_only=True,
        )
    )

    baseline_model = (
        BartForConditionalGeneration
        .from_pretrained(
            BART_LOCAL,
            local_files_only=True,
            use_safetensors=True,
            low_cpu_mem_usage=True,
        )
    )

    def to_dataset(
        split_examples,
    ):
        return Dataset.from_dict({
            'linearized': [
                example['linearized']
                for example
                in split_examples
            ],
            'target': [
                example['target']
                for example
                in split_examples
            ],
        })

    def preprocess(batch):
        encoded = baseline_tokenizer(
            batch['linearized'],
            max_length=MAX_INPUT_LEN,
            truncation=True,
        )

        labels = baseline_tokenizer(
            text_target=batch['target'],
            max_length=MAX_TARGET_LEN,
            truncation=True,
        )

        encoded['labels'] = labels[
            'input_ids'
        ]
        return encoded

    train_dataset = (
        to_dataset(data['train'])
        .map(
            preprocess,
            batched=True,
            remove_columns=[
                'linearized',
                'target',
            ],
            desc=(
                'Tokenising baseline train'
            ),
        )
    )

    dev_dataset = (
        to_dataset(data['dev'])
        .map(
            preprocess,
            batched=True,
            remove_columns=[
                'linearized',
                'target',
            ],
            desc=(
                'Tokenising baseline dev'
            ),
        )
    )

    collator = DataCollatorForSeq2Seq(
        tokenizer=baseline_tokenizer,
        model=baseline_model,
    )

    training_args = (
        Seq2SeqTrainingArguments(
            output_dir=str(
                baseline_path
            ),
            num_train_epochs=(
                BASELINE_EPOCHS
            ),
            per_device_train_batch_size=(
                BASELINE_BATCH_SIZE
            ),
            per_device_eval_batch_size=(
                BASELINE_BATCH_SIZE
                * 2
            ),
            learning_rate=(
                BASELINE_LR
            ),
            lr_scheduler_type='constant',
            warmup_steps=0,
            weight_decay=0.01,
            fp16=torch.cuda.is_available(),
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model=(
                'eval_loss'
            ),
            greater_is_better=False,
            save_total_limit=2,
            logging_steps=200,
            label_smoothing_factor=0.0,
            predict_with_generate=False,
            report_to='none',
            seed=SEED,
        )
    )

    trainer = Seq2SeqTrainer(
        model=baseline_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        data_collator=collator,
    )

    last_checkpoint = (
        get_last_checkpoint(
            str(baseline_path)
        )
    )

    if last_checkpoint:
        print(
            'Resuming baseline training from:',
            last_checkpoint,
        )

        trainer.train(
            resume_from_checkpoint=(
                last_checkpoint
            )
        )
    else:
        trainer.train()

    trainer.save_model(
        str(baseline_path)
    )

    baseline_tokenizer.save_pretrained(
        str(baseline_path)
    )

    with open(
        baseline_path
        / 'training_done.flag',
        'w',
        encoding='utf-8',
    ) as handle:
        handle.write('ok')

    print(
        'Saved rebuilt fair baseline to:',
        baseline_path,
    )

    return trainer.model.to(
        DEVICE
    )

model_base = (
    _train_or_load_fair_baseline()
)
model_base.eval()

(
    tokenizer,
    bart_shared,
) = load_pretrained_bart(
    BART_LOCAL,
    device=DEVICE,
    train_bart=False,
)

model_full = (
    FullDualOutputGNNModel(
        bart_shared,
        num_relations=(
            num_relations
        ),
    )
    .to(DEVICE)
)

model_full = (
    load_variant_checkpoint_resume(
        model_full,
        CKPT_FULL,
        DEVICE,
        strict=False,
    )
)
model_full.eval()

bart_for_fusion = (
    BartForConditionalGeneration
    .from_pretrained(
        BART_LOCAL,
        local_files_only=True,
        use_safetensors=True,
        low_cpu_mem_usage=True,
    )
    .to(DEVICE)
)

model_fusion = (
    FusionOnlyGNNModel(
        bart_for_fusion,
        num_relations=(
            num_relations
        ),
    )
    .to(DEVICE)
)

model_fusion = (
    load_variant_checkpoint_resume(
        model_fusion,
        CKPT_FUSION,
        DEVICE,
        strict=False,
    )
)
model_fusion.eval()

for model in (
    model_full,
    model_fusion,
    model_base,
):
    for parameter in model.parameters():
        parameter.requires_grad_(
            False
        )

torch.set_grad_enabled(
    False
)
torch.cuda.empty_cache()

allocated_gb = (
    torch.cuda.memory_allocated()
    / 1e9
)

print(
    f'models ready | GPU allocated: '
    f'{allocated_gb:.2f} GB'
)


In [ ]:
# 4. Manifest + metrics + artifact audit (identical protocol to notebooks 9/12).
from collections import OrderedDict
import re, unicodedata
from difflib import SequenceMatcher
import sacrebleu

def _tp(t):
    if isinstance(t, dict): return str(t.get('subject','')), str(t.get('predicate','')), str(t.get('object',''))
    return str(t[0]), str(t[1]), str(t[2])

groups = OrderedDict()
for i, ex in enumerate(data['test']):
    key = json.dumps([_tp(t) for t in ex['triples']], ensure_ascii=False)
    g = groups.setdefault(key, {'first_index': i, 'references': []})
    for v in list(ex.get('all_targets') or []) + [ex.get('target','')]:
        v = str(v).strip()
        if v and v not in g['references']: g['references'].append(v)
train_predicates = {_tp(t)[1] for ex in data['train'] for t in ex['triples']}
ITEMS = []
for uid, (key, g) in enumerate(groups.items()):
    ex = dict(data['test'][g['first_index']]); ex['all_targets'] = g['references']
    ITEMS.append({'uid': uid, 'idx': g['first_index'], 'ex': ex,
                  'unseen': bool({_tp(t)[1] for t in ex['triples']} - train_predicates)})
assert len(ITEMS) == 2510 and sum(x['unseen'] for x in ITEMS) == 752
print('manifest ok')

_TR = {'ø':'o','Ø':'o','æ':'ae','Æ':'ae','œ':'oe','Œ':'oe','ð':'d','Ð':'d','þ':'th','Þ':'th',
       'ł':'l','Ł':'l','ß':'ss','đ':'d','Đ':'d','ħ':'h','ı':'i','İ':'i','ŋ':'ng'}
_MONTHS = ['january','february','march','april','may','june','july','august','september','october','november','december']
_DET = re.compile(r'^(the|a|an)\s+')
_STOP = frozenset(('the a an and or but if then this that these those it its he she they them his her their '
                   'in on at of to by with for from as is are was were be been being there here also however').split())
def _sa(s):
    s = ''.join(_TR.get(c, c) for c in str(s))
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))
def _norm(s):
    s = _sa(str(s)).lower().strip().strip('"').strip("'")
    s = re.sub(r'[^a-z0-9 ]', ' ', s); s = re.sub(r'\s+', ' ', s).strip()
    return _DET.sub('', s)
def _wm(t, s):
    return bool(s) and re.search(r'(?<![a-z0-9])' + re.escape(s) + r'(?![a-z0-9])', t) is not None
def _dst(v):
    toks = set(); raw = _sa(str(v)).strip().strip('"').strip("'")
    m = re.match(r'^(\d{3,4})-(\d{1,2})-(\d{1,2})$', raw)
    if m:
        y, mo, d = map(int, m.groups()); toks.add(str(y))
        if 1 <= mo <= 12: toks.add(_MONTHS[mo-1]); toks.add(_MONTHS[mo-1][:3])
        toks.update({str(d), str(d).zfill(2)}); toks.update({f'{d}{sx}' for sx in ('st','nd','rd','th')})
    elif re.match(r'^\d{3,4}$', raw): toks.add(raw)
    return toks
def _dg(s): return re.sub(r'[^0-9]', '', str(s))
def _grounding(triples):
    forms, tokens, num = [], set(), set()
    for t in triples:
        s, p, o = _tp(t)
        for v in (s, o):
            n = _norm(v)
            if n: forms.append(n); tokens.update(n.split())
            tokens.update(_dst(v))
            d = _dg(v)
            if d: num.add(d)
        tokens.update(_norm(re.sub(r'([a-z])([A-Z])', r'\1 \2', p)).split())
    return forms, tokens, num
def _mentions(text):
    men = set()
    for m in re.findall(r'[A-Z][A-Za-z]*(?:[ -][A-Z][A-Za-z]*)*', _sa(text)):
        n = _norm(m)
        if len(n) >= 3 and (' ' in n or n not in _STOP): men.add(n)
    for m in re.findall(r'[A-Za-z0-9]+(?:[./\-][A-Za-z0-9]+)*', str(text)):
        if any(c.isdigit() for c in m): men.add(_norm(m))
    return {m for m in men if m}
def grounding_score(prediction, triples):
    pn = _norm(prediction)
    forms, tokens, num = _grounding(triples)
    fd = [f.replace(' ', '') for f in forms]
    found = sum(1 for f in forms if _wm(pn, f) or all(_wm(pn, w) for w in f.split()))
    recall = found / len(forms) if forms else 1.0
    def ok(m):
        for f in forms:
            if m == f or _wm(f, m): return True
        if all(t in tokens for t in m.split()): return True
        d = _dg(m)
        if d and any(d in c or c in d for c in num): return True
        md = m.replace(' ', '')
        return len(md) >= 3 and any(md in x or x in md for x in fd)
    mens = _mentions(prediction)
    hall = sorted(m for m in mens if not ok(m))
    corr = [m for m in hall if max((SequenceMatcher(None, m, f).ratio() for f in forms), default=0) >= 0.55]
    return {'halluc': (len(hall) / len(mens) if mens else 0.0),
            'recall': recall, 'hallucinated': hall, 'corruptions': corr}

ARTIFACT_PATS = {
    'iso_date_midtext': r'[A-Za-z],? \d{3,4}-\d{1,2}-\d{1,2}|\d{1,2}(st|nd|rd|th)? [A-Za-z]+ \d{3,4}-\d{1,2}-\d{1,2}',
    'paren_disambig': r'\((The [^)]+album|[0-9]{4} film|film|band|song|album|actor[^)]*|footballer[^)]*|musician[^)]*)\)',
    'raw_unit_paren': r'\d[\d.,]*\s*\((milli|centi|kilo)?(metres|meters|grams|litres|liters|inches)\)',
}
def artifact_flags(pred):
    return {k: bool(re.search(p, str(pred))) for k, p in ARTIFACT_PATS.items()}

def corpus_bleu_lc(preds, refs_list):
    maxr = max(len(r) for r in refs_list)
    streams = [[r[k] if k < len(r) else r[0] for r in refs_list] for k in range(maxr)]
    return sacrebleu.corpus_bleu(preds, streams, lowercase=True, tokenize='13a').score
print('metrics + artifact audit ready')


In [ ]:
# 5. TrieMapV2 — cleaned name-only completion trie, with self-tests.
class TNode:
    __slots__ = ('ch', 'score', 'mx', 'nterm', 'terminal')
    def __init__(self):
        self.ch = {}; self.score = None; self.mx = -1e9; self.nterm = 0; self.terminal = False

_LITERAL_PATS = [r'^[\d\s.,:/\-+%°"]*$', r'^\d{3,4}-\d{1,2}-\d{1,2}', r'^\d+(\.\d+)?$']
def is_literal(name):
    n = str(name).strip().strip('"').strip("'").strip()
    if len(n) < 2: return True
    return any(re.match(p, n) for p in _LITERAL_PATS)

def clean_surface(name):
    s = str(name).strip().strip('"').strip("'").replace('_', ' ')
    s = re.sub(r'\s*\([^)]*\)', ' ', s)           # strip DBpedia disambiguation parentheticals
    s = re.sub(r'\s+', ' ', s).strip()
    return s

class TrieMapV2:
    def __init__(self, entity_names, tokenizer, entity_scores=None):
        self.root = TNode(); self.kept = []
        scores = entity_scores if entity_scores is not None else [0.0] * len(entity_names)
        for name, s in zip(entity_names, scores):
            if is_literal(name): continue
            base = clean_surface(name)
            if not base or is_literal(base): continue
            self.kept.append(base)
            s = float(s.item() if hasattr(s, 'item') else s)
            for v in (base, ' ' + base):
                ids = tokenizer.encode(v, add_special_tokens=False)
                if not ids: continue
                n = self.root
                for tid in ids: n = n.ch.setdefault(int(tid), TNode())
                if n.score is None or abs(s) > abs(n.score): n.score = s
                n.terminal = True
        self._cache(self.root)
    def _cache(self, n):
        mx = n.score if n.score is not None else -1e9
        nt = 1 if n.terminal else 0
        for c in n.ch.values():
            self._cache(c); mx = max(mx, c.mx); nt += c.nterm
        n.mx = mx; n.nterm = nt
    def suffix_matches(self, gen_ids, lookback=20):
        out = []
        for start in range(max(0, len(gen_ids) - lookback), len(gen_ids)):
            n = self.root; ok = True
            for pos in range(start, len(gen_ids)):
                t = int(gen_ids[pos])
                if t not in n.ch: ok = False; break
                n = n.ch[t]
            if ok and n is not self.root:
                out.append((len(gen_ids) - start, n))
        return out

def hard_mask_v2(logits, trie, gen_ids, score_min=None):
    best = None
    for depth, node in trie.suffix_matches(gen_ids):
        if node.terminal or not node.ch: continue      # boundary release: completed names impose nothing
        trig = depth >= LOCK_MIN_DEPTH or node.nterm == 1
        if score_min is not None: trig = trig and node.mx >= score_min
        if trig and (best is None or depth > best[0]): best = (depth, node)
    if best is None: return logits, False
    legal = list(best[1].ch.keys())
    if float(logits.max()) - max(float(logits[t]) for t in legal) > ESCAPE_MARGIN:
        return logits, False
    m = torch.full_like(logits, -1e9); m[legal] = 0.0
    return logits + m, True

# self-tests
_t = TrieMapV2(['Pontiac_Rageous', '1964-10-13', '11.5', 'Harry_Carey_(actor_born_1878)', '"Alvinegro"'], tokenizer)
assert 'Pontiac Rageous' in _t.kept and 'Harry Carey' in _t.kept and 'Alvinegro' in _t.kept
assert not any('1964' in k or '11.5' in k for k in _t.kept)
ids_hc = tokenizer.encode(' Harry Carey', add_special_tokens=False)
n = _t.root
for tid in ids_hc: n = n.ch[tid]
assert n.terminal and not n.ch, 'parenthetical was not stripped from the trie path'
print('TrieMapV2 self-tests passed | kept:', _t.kept)


In [ ]:
# 6. Decoders: GNN models (fusion / full_dual) and plain BART, off | hard_v2 | posonly | hard_v2_gated.
@torch.no_grad()
def decode_gnn(model, ex, graph, mode='off', alpha=1.0, score_min=None):
    enc_in = tokenizer(ex['linearized'], max_length=MAX_INPUT_LEN, truncation=True,
                       padding='max_length', return_tensors='pt')
    inp, att = enc_in['input_ids'].to(DEVICE), enc_in['attention_mask'].to(DEVICE)
    gb = torch.zeros(graph.x.size(0), dtype=torch.long, device=DEVICE)
    enc = model.bart.model.encoder(input_ids=inp, attention_mask=att)
    h_kg, c_ent = model.rgcn(graph.x.to(DEVICE), graph.edge_index.to(DEVICE), graph.edge_type.to(DEVICE))
    scores = signed_entity_scores(c_ent).detach().cpu()
    names = list(getattr(graph, 'entity_names', []) or [])
    trie = None
    if mode in ('hard_v2', 'hard_v2_gated'): trie = TrieMapV2(names, tokenizer, scores)
    elif mode == 'posonly': trie = TrieMapV2(names, tokenizer, torch.clamp(scores, min=0.0))
    h_pad, k_mask = pad_kg_nodes(h_kg, gb, 1)
    dec_start, eos = model.bart.config.decoder_start_token_id, model.bart.config.eos_token_id
    gen = [dec_start]; past = None; forced = 0
    for _ in range(MAX_GEN_LEN - 1):
        di = torch.tensor([[gen[-1]]], dtype=torch.long, device=DEVICE)
        out = model.bart.model.decoder(input_ids=di, encoder_hidden_states=enc.last_hidden_state,
                                       encoder_attention_mask=att, past_key_values=past, use_cache=True)
        past = out.past_key_values
        h, _, _ = model.kg_cross_attention(out.last_hidden_state, h_pad, k_mask)
        logits = add_final_logits_bias(model.bart, model.bart.lm_head(h))[:, -1, :].squeeze(0).float().cpu()
        free = int(logits.argmax())
        if mode in ('hard_v2', 'hard_v2_gated'):
            logits, _ = hard_mask_v2(logits, trie, gen[1:], score_min=(0.0 if mode == 'hard_v2_gated' else None))
        elif mode == 'posonly':
            c = torch.zeros(len(tokenizer))
            for tid, ch in trie.root.ch.items():
                sub = ch.mx if ch.mx > -1e8 else 0.0
                c[tid] += max(sub, 0.0)
            for depth, node in trie.suffix_matches(gen[1:]):
                if node.ch:
                    for tid, ch in node.ch.items(): c[tid] += 2.0 * max(ch.mx, 0.0)
            logits = logits + alpha * c
        nxt = int(logits.argmax())
        if nxt != free: forced += 1
        if nxt == eos: break
        gen.append(nxt)
    return tokenizer.decode(gen[1:], skip_special_tokens=True), forced

@torch.no_grad()
def decode_bart(model, ex, graph, mode='off'):
    enc_in = tokenizer(ex['linearized'], max_length=MAX_INPUT_LEN, truncation=True,
                       padding='max_length', return_tensors='pt')
    inp, att = enc_in['input_ids'].to(DEVICE), enc_in['attention_mask'].to(DEVICE)
    enc = model.model.encoder(input_ids=inp, attention_mask=att)
    trie = TrieMapV2(list(getattr(graph, 'entity_names', []) or []), tokenizer) if mode == 'hard_v2' else None
    dec_start, eos = model.config.decoder_start_token_id, model.config.eos_token_id
    gen = [dec_start]; past = None; forced = 0
    for _ in range(MAX_GEN_LEN - 1):
        di = torch.tensor([[gen[-1]]], dtype=torch.long, device=DEVICE)
        out = model.model.decoder(input_ids=di, encoder_hidden_states=enc.last_hidden_state,
                                  encoder_attention_mask=att, past_key_values=past, use_cache=True)
        past = out.past_key_values
        logits = model.lm_head(out.last_hidden_state)
        if getattr(model, 'final_logits_bias', None) is not None:
            logits = logits + model.final_logits_bias.to(logits.device)
        logits = logits[:, -1, :].squeeze(0).float().cpu()
        free = int(logits.argmax())
        if trie is not None:
            logits, _ = hard_mask_v2(logits, trie, gen[1:])
        nxt = int(logits.argmax())
        if nxt != free: forced += 1
        if nxt == eos: break
        gen.append(nxt)
    return tokenizer.decode(gen[1:], skip_special_tokens=True), forced

# regression: full_dual off must reproduce notebook-12 A_off where available
ref_off = None
p = os.path.join(NB12_EVAL, 'preds_A_off.json')
if os.path.exists(p): ref_off = json.load(open(p))
for j in (0, 7):
    it = ITEMS[j]
    mine, _ = decode_gnn(model_full, it['ex'], graphs['test'][it['idx']], 'off')
    if ref_off: assert mine == ref_off[j], f'regression mismatch at {j}'
print('regression', 'passed vs preds_A_off' if ref_off else 'skipped (preds_A_off not found)')


In [ ]:
# 7. Main runs: three systems × {off, hard_v2}, with atomic incremental resume.
SUMMARY_PATH = os.path.join(EVAL_OUT, 'hard_v2_summary.json')
SUMMARY = json.load(open(SUMMARY_PATH, encoding='utf-8')) if os.path.exists(SUMMARY_PATH) else {}


def _load_list(path):
    if not os.path.exists(path):
        return []
    try:
        obj = json.load(open(path, encoding='utf-8'))
        if not isinstance(obj, list):
            raise ValueError('expected a JSON list')
        return obj
    except Exception as exc:
        broken = path + '.broken'
        shutil.move(path, broken)
        print(f'Corrupt partial file moved to {broken}: {exc}')
        return []


def run(tag, fn):
    pred_path = os.path.join(EVAL_OUT, f'preds_{tag}.json')
    forced_path = os.path.join(EVAL_OUT, f'forced_counts_{tag}.json')
    preds = _load_list(pred_path)
    forced_counts = _load_list(forced_path)

    if len(preds) > len(ITEMS):
        raise RuntimeError(f'{pred_path} has too many rows ({len(preds)} > {len(ITEMS)}).')
    if len(forced_counts) > len(preds):
        forced_counts = forced_counts[:len(preds)]
    if len(forced_counts) < len(preds):
        forced_counts.extend([None] * (len(preds) - len(forced_counts)))

    start = len(preds)
    if start:
        print(f'[{tag}] resuming at {start}/{len(ITEMS)}')
    else:
        print(f'[{tag}] starting')
    t0 = time.time()

    for i in range(start, len(ITEMS)):
        pred, forced = fn(ITEMS[i])
        preds.append(pred)
        forced_counts.append(int(forced))
        if (i + 1) % SAVE_EVERY == 0 or (i + 1) == len(ITEMS):
            atomic_json_dump(preds, pred_path)
            atomic_json_dump(forced_counts, forced_path)
        if (i + 1) % 250 == 0:
            print(f'  [{tag}] {i+1}/{len(ITEMS)} | this session {time.time()-t0:.0f}s')

    return preds


def score(preds):
    assert len(preds) == len(ITEMS), f'Expected {len(ITEMS)} predictions, got {len(preds)}'
    refs = [it['ex']['all_targets'] for it in ITEMS]
    gs = [grounding_score(p, it['ex']['triples']) for p, it in zip(preds, ITEMS)]
    art = [artifact_flags(p) for p in preds]
    return {
        'n': len(preds),
        'bleu': corpus_bleu_lc(preds, refs),
        'halluc': float(np.mean([g['halluc'] for g in gs])),
        'recall': float(np.mean([g['recall'] for g in gs])),
        'corr_rows': int(np.sum([len(g['corruptions']) > 0 for g in gs])),
        'art_iso': int(np.sum([a['iso_date_midtext'] for a in art])),
        'art_paren': int(np.sum([a['paren_disambig'] for a in art])),
        'art_unit': int(np.sum([a['raw_unit_paren'] for a in art])),
    }


# Reuse only complete, correctly sized off-prediction files.
OFF_REUSE_CANDIDATES = {
    'full': [os.path.join(NB12_EVAL, 'preds_A_off.json')],
    'baseline': [os.path.join(BASE_EVAL, 'predictions_baseline_test.json')],
    'fusion': [
        os.path.join(NB12_EVAL, 'preds_fusion_off.json'),
        os.path.join(NB12_EVAL, 'preds_A_fusion_off.json'),
    ],
}

PREDS = {}
systems = [
    ('baseline',
     lambda it: decode_bart(model_base, it['ex'], graphs['test'][it['idx']], 'off'),
     lambda it: decode_bart(model_base, it['ex'], graphs['test'][it['idx']], 'hard_v2')),
    ('fusion',
     lambda it: decode_gnn(model_fusion, it['ex'], graphs['test'][it['idx']], 'off'),
     lambda it: decode_gnn(model_fusion, it['ex'], graphs['test'][it['idx']], 'hard_v2')),
    ('full',
     lambda it: decode_gnn(model_full, it['ex'], graphs['test'][it['idx']], 'off'),
     lambda it: decode_gnn(model_full, it['ex'], graphs['test'][it['idx']], 'hard_v2')),
]

for sysname, fn_off, fn_hard in systems:
    off_tag = f'{sysname}_off'
    off_dest = os.path.join(EVAL_OUT, f'preds_{off_tag}.json')
    if not os.path.exists(off_dest):
        for candidate in OFF_REUSE_CANDIDATES.get(sysname, []):
            if os.path.exists(candidate):
                loaded = _load_list(candidate)
                if len(loaded) == len(ITEMS):
                    atomic_json_dump(loaded, off_dest)
                    print(f'[{off_tag}] reused from {candidate}')
                    break

    PREDS[off_tag] = run(off_tag, fn_off)
    PREDS[f'{sysname}_hard_v2'] = run(f'{sysname}_hard_v2', fn_hard)

    for tag in (off_tag, f'{sysname}_hard_v2'):
        SUMMARY[tag] = score(PREDS[tag])
        atomic_json_dump(SUMMARY, SUMMARY_PATH, indent=2)
        print(tag, SUMMARY[tag])


In [ ]:
# 8. Paired deltas + newly-introduced artifact rows (the honest table).
rng = np.random.default_rng(0)
def paired_delta(pa, pb, fn):
    a = np.array([fn(x, it) for x, it in zip(pa, ITEMS)], dtype=float)
    b = np.array([fn(x, it) for x, it in zip(pb, ITEMS)], dtype=float)
    d = a - b
    boot = [rng.choice(d, len(d), replace=True).mean() for _ in range(2000)]
    return d.mean(), np.percentile(boot, 2.5), np.percentile(boot, 97.5)

print('%-9s | d_halluc (CI) | d_recall (CI) | corr_rows off->hard | new-artifact rows | changed%%' % 'system')
for sysname in ('baseline', 'fusion', 'full'):
    po, ph = PREDS[f'{sysname}_off'], PREDS[f'{sysname}_hard_v2']
    dh = paired_delta(ph, po, lambda p, it: grounding_score(p, it['ex']['triples'])['halluc'])
    dr = paired_delta(ph, po, lambda p, it: grounding_score(p, it['ex']['triples'])['recall'])
    ao = [any(artifact_flags(p).values()) for p in po]
    ah = [any(artifact_flags(p).values()) for p in ph]
    new_art = int(np.sum([(not a) and b for a, b in zip(ao, ah)]))
    ch = 100 * float(np.mean([a != b for a, b in zip(po, ph)]))
    print('%-9s | %+.4f [%.4f,%.4f] | %+.4f [%.4f,%.4f] | %d -> %d | %d | %.1f%%' % (
        sysname, dh[0], dh[1], dh[2], dr[0], dr[1], dr[2],
        SUMMARY[f'{sysname}_off']['corr_rows'], SUMMARY[f'{sysname}_hard_v2']['corr_rows'], new_art, ch))


In [ ]:
# 9. Held-out dev validation of the two test-exposed variants, also resume-safe.
if RUN_HELDOUT_VALIDATION:
    rng = random.Random(7)
    tuned_idx = set(rng.sample(range(len(data['dev'])), 300))
    pool = [i for i in range(len(data['dev'])) if i not in tuned_idx]
    held = sorted(random.Random(21).sample(pool, HELDOUT_N))
    H_ITEMS = []
    for i in held:
        ex = dict(data['dev'][i])
        ex['all_targets'] = [
            str(t).strip() for t in (ex.get('all_targets') or [ex.get('target', '')])
            if str(t).strip()
        ]
        H_ITEMS.append({'idx': i, 'ex': ex})

    held_manifest_path = os.path.join(EVAL_OUT, 'heldout_dev_indices.json')
    atomic_json_dump(held, held_manifest_path)

    def run_h(tag, mode, **kw):
        path = os.path.join(EVAL_OUT, f'preds_heldout_{tag}.json')
        preds = _load_list(path)
        if len(preds) > len(H_ITEMS):
            raise RuntimeError(f'{path} has too many rows.')
        start = len(preds)
        print(f'[heldout_{tag}] resume {start}/{len(H_ITEMS)}')
        for j in range(start, len(H_ITEMS)):
            it = H_ITEMS[j]
            pred, _ = decode_gnn(model_full, it['ex'], graphs['dev'][it['idx']], mode, **kw)
            preds.append(pred)
            if (j + 1) % SAVE_EVERY == 0 or (j + 1) == len(H_ITEMS):
                atomic_json_dump(preds, path)
        return preds

    def score_h(preds):
        refs = [it['ex']['all_targets'] for it in H_ITEMS]
        gs = [grounding_score(p, it['ex']['triples']) for p, it in zip(preds, H_ITEMS)]
        art = [artifact_flags(p) for p in preds]
        return {
            'n': len(preds),
            'bleu': corpus_bleu_lc(preds, refs),
            'halluc': float(np.mean([g['halluc'] for g in gs])),
            'recall': float(np.mean([g['recall'] for g in gs])),
            'corr_rows': int(np.sum([len(g['corruptions']) > 0 for g in gs])),
            'artifact_rows': int(np.sum([any(a.values()) for a in art])),
        }

    held_modes = [
        ('off', 'off', {}),
        ('posonly_a1', 'posonly', {'alpha': 1.0}),
        ('hard_v2', 'hard_v2', {}),
        ('hard_v2_gated', 'hard_v2_gated', {}),
    ]
    for tag, mode, kw in held_modes:
        s = score_h(run_h(tag, mode, **kw))
        SUMMARY[f'heldout_{tag}'] = s
        atomic_json_dump(SUMMARY, SUMMARY_PATH, indent=2)
        print('heldout', tag, s)


In [ ]:
# 10. Final table + one ZIP file for upload.
import pandas as pd

rows = [{'tag': tag, **metrics} for tag, metrics in SUMMARY.items()]
frame = pd.DataFrame(rows).sort_values('tag').reset_index(drop=True)
flat_path = os.path.join(EVAL_OUT, 'hard_v2_summary_flat.csv')
frame.to_csv(flat_path, index=False)
display(frame)

zip_path = shutil.make_archive(EVAL_OUT, 'zip', EVAL_OUT)
print('\nRUN COMPLETE')
print('Result folder:', EVAL_OUT)
print('Single file to upload:', zip_path)
print('\nFiles:')
for f in sorted(os.listdir(EVAL_OUT)):
    print('  ', f)
